In [ ]:
# Imports

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [3]:
# Input paths
paths = {
    'SPECTER': r"data/specter_embeddings.csv", 
        'BERT': r"data/bert_embeddings.csv",        
        'SciBERT': r"data/scibert_embeddings.csv",  
        'target': r"data/targets.csv",                
    "Dataset":r"data/combined_acl_arxiv_with_citations_final.csv",
    "split":r"train_val_test_split.json"
}

In [4]:
# Load dataset, SPECTER embeddings and targets
data = pd.read_csv(paths["Dataset"])
specter = pd.read_csv(paths["SPECTER"])
targets = pd.read_csv(paths["target"])

print("Dataset shape:", data.shape)
print("Specter shape:", specter.shape)
print("Targets shape:", targets.shape)

Dataset shape: (131772, 9)
Specter shape: (131772, 770)
Targets shape: (131772, 2)


In [5]:
# Merge dataset + SPECTER + targets on project_id
df = data.merge(specter, on="project_id", how="inner")
df = df.merge(targets, on="project_id", how="inner")

print("Merged shape:", df.shape)

Merged shape: (131772, 779)


In [6]:
# Stylometric feature extractor (word/sentence stats, lexical diversity)

import re
import textstat

def compute_stylometric_features(text):
    
    if pd.isna(text):
        text = ""
        
    words = re.findall(r'\b\w+\b', text)
    sentences = re.split(r'[.!?\n]+', text)    
    word_count = len(words)
    sentence_count = len([s for s in sentences if len(s.strip()) > 0])
    
    avg_word_length = np.mean([len(w) for w in words]) if word_count > 0 else 0
    avg_sentence_length = word_count / sentence_count if sentence_count > 0 else 0
    
    lexical_diversity = len(set(words)) / word_count if word_count > 0 else 0
    
    complex_words = [w for w in words if len(w) >= 7]
    complex_word_ratio = len(complex_words) / word_count if word_count > 0 else 0
    
    # Readability metrics
    # flesch_reading_ease = textstat.flesch_reading_ease(text) if word_count > 0 else 0
    # flesch_kincaid_grade = textstat.flesch_kincaid_grade(text) if word_count > 0 else 0
    
    return pd.Series({
        "word_count": word_count,
        "sentence_count": sentence_count,
        "avg_word_length": avg_word_length,
        "avg_sentence_length": avg_sentence_length,
        "lexical_diversity": lexical_diversity,
        "complex_word_ratio": complex_word_ratio,
        # "flesch_reading_ease": flesch_reading_ease,
        # "flesch_kincaid_grade": flesch_kincaid_grade
    })

In [7]:
# Build combined title+abstract text field
df["combined_text"] = (
    df["title"].fillna("") + " " + df["abstract"].fillna("")
)

In [8]:
# Compute stylometric features
stylometric_df = df["combined_text"].apply(compute_stylometric_features)

In [9]:
# Summary of stylometric features
stylometric_df.describe()

,word_count,sentence_count,avg_word_length,avg_sentence_length,lexical_diversity,complex_word_ratio
count,131772.000000,131772.000000,131772.000000,131772.000000,131772.000000,131772.000000
mean,159.807683,14.732212,5.582162,15.632253,0.678372,0.376624
std,57.496980,9.853838,0.412429,9.583078,0.072513,0.060867
min,9.000000,1.000000,1.112281,1.576923,0.078363,0.000000
25%,120.000000,6.000000,5.328671,8.000000,0.631579,0.336898
50%,157.000000,12.000000,5.579545,9.600000,0.676471,0.376238
75%,194.000000,23.000000,5.839506,22.500000,0.723881,0.416185
max,1462.000000,264.000000,8.750000,261.000000,1.000000,0.680365


In [10]:
# Attach stylometric feature columns
stylometric_cols = [
    "word_count",
    "sentence_count",
    "avg_word_length",
    "avg_sentence_length",
    "lexical_diversity",
    "complex_word_ratio"
]

df = pd.concat([df, stylometric_df[stylometric_cols]], axis=1)

print("Updated df shape:", df.shape)

Updated df shape: (131772, 786)


In [ ]:
# Helper: load the id split and slice df (optionally fold val into train)
import json
def get_splits(df,val_split=True):
    with open(paths["split"], "r") as f:
        split_dict = json.load(f)

    train_ids = set(split_dict["train"])
    val_ids   = set(split_dict["val"])
    test_ids  = set(split_dict["test"])
    if val_split:
        train_df = df[df["project_id"].isin(train_ids)].copy()
        val_df   = df[df["project_id"].isin(val_ids)].copy()
        test_df  = df[df["project_id"].isin(test_ids)].copy()
        return train_df, val_df, test_df, train_ids, val_ids, test_ids
    else:
        train_ids = set(split_dict["train"]) |set(split_dict["val"])
        test_ids  = set(split_dict["test"])
        train_df = df[df["project_id"].isin(train_ids)].copy()
        test_df  = df[df["project_id"].isin(test_ids)].copy()
        return train_df, test_df, train_ids, test_ids
    
train_df, val_df, test_df, train_ids, val_ids, test_ids = get_splits(df)
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) == len(train_ids)
assert len(val_df) == len(val_ids)
assert len(test_df) == len(test_ids)

print("Split integrity passed.")

Train: (104685, 786)
Val: (20129, 786)
Test: (6958, 786)
Split integrity passed.


In [12]:
# Column list
df.columns

Index(['project_id', 'original_id', 'abstract', 'citation_count',
       'publication_year', 'title', 'authors_str', 'source',
       'original_id_clean_x', 'original_id_clean_y',
       ...
       'specter_embedding_766', 'specter_embedding_767', 'is_highly_cited',
       'combined_text', 'word_count', 'sentence_count', 'avg_word_length',
       'avg_sentence_length', 'lexical_diversity', 'complex_word_ratio'],
      dtype='object', length=786)

In [13]:
# Collect the SPECTER feature columns
specter_cols = [c for c in df.columns if c.startswith("specter_")]

print("Specter feature count:", len(specter_cols))

Specter feature count: 768


In [14]:
# Feature set: SPECTER + stylometric (+ target)
TARGET_COL = "is_highly_cited"

MODEL_COLS = specter_cols + stylometric_cols + [TARGET_COL]

In [ ]:
# Helper: build X/y matrices (with or without a val split)
def get_train_val_test_data(train_df, test_df, MODEL_COLS, TARGET_COL,val_df=None):
    if val_df is not None:
        X_train = train_df[MODEL_COLS].drop(columns=[TARGET_COL])
        y_train = train_df[TARGET_COL]

        X_val = val_df[MODEL_COLS].drop(columns=[TARGET_COL])
        y_val = val_df[TARGET_COL]

        X_test = test_df[MODEL_COLS].drop(columns=[TARGET_COL])
        y_test = test_df[TARGET_COL]
        return X_train, y_train, X_val, y_val, X_test, y_test
    else:
        X_train = train_df[MODEL_COLS].drop(columns=[TARGET_COL])
        y_train = train_df[TARGET_COL]

        X_test = test_df[MODEL_COLS].drop(columns=[TARGET_COL])
        y_test = test_df[TARGET_COL]
        return X_train, y_train, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = get_train_val_test_data(train_df=train_df, val_df=val_df, test_df=test_df, MODEL_COLS=MODEL_COLS, TARGET_COL=TARGET_COL)

print("X_train shape:", X_train.shape)

X_train shape: (104685, 774)


In [16]:
# Class balance of the training set
y_train.value_counts()

is_highly_cited
0    83360
1    21325
Name: count, dtype: int64

In [17]:
# Class weights to offset the imbalance
from collections import Counter

counter = Counter(y_train)
total = len(y_train)

class_weights = {
    0: total / (2 * counter[0]),
    1: total / (2 * counter[1])
}

print("Class weights:", class_weights)

Class weights: {0: 0.6279090690978887, 1: 2.4545134818288394}


In [18]:
# CatBoost reference docs
from catboost import CatBoostClassifier
help(CatBoostClassifier)

Help on class CatBoostClassifier in module catboost.core:

class CatBoostClassifier(CatBoost)
 |  CatBoostClassifier(
 |      iterations=None,
 |      learning_rate=None,
 |      depth=None,
 |      l2_leaf_reg=None,
 |      model_size_reg=None,
 |      rsm=None,
 |      loss_function=None,
 |      border_count=None,
 |      feature_border_type=None,
 |      per_float_feature_quantization=None,
 |      input_borders=None,
 |      output_borders=None,
 |      fold_permutation_block=None,
 |      od_pval=None,
 |      od_wait=None,
 |      od_type=None,
 |      nan_mode=None,
 |      counter_calc_method=None,
 |      leaf_estimation_iterations=None,
 |      leaf_estimation_method=None,
 |      thread_count=None,
 |      random_seed=None,
 |      use_best_model=None,
 |      best_model_min_trees=None,
 |      verbose=None,
 |      silent=None,
 |      logging_level=None,
 |      metric_period=None,
 |      ctr_leaf_count_limit=None,
 |      store_all_simple_ctr=None,
 |      max_ctr_compl

In [19]:
# train_and_evaluate with tunable hyperparameters
import numpy as np
import json
import joblib
from catboost import CatBoostClassifier
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

def train_and_evaluate(
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    model_name,
    params=None,
    save_dir="results"
):
    default_params = {
    "iterations": 2000,
    "learning_rate": 0.05,
    "depth": 6,
    "l2_leaf_reg": 3,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": 42,
    "early_stopping_rounds": 100,
    "verbose": False
}

    if params is not None:
        default_params.update(params)

    model = CatBoostClassifier(**default_params)
    print(f"\n===== Training {model_name} =====")

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        early_stopping_rounds=100,
        verbose=200
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True
    )

    # --- Probabilities ---
    val_probs = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_test)[:, 1]

    # --- PR-AUC ---
    val_pr_auc = average_precision_score(y_val, val_probs)
    test_pr_auc = average_precision_score(y_test, test_probs)

    # --- Threshold tuning (VALIDATION ONLY) ---
    thresholds = np.linspace(0.05, 0.95, 200)

    best_thr = 0.5
    best_f1 = 0

    for t in thresholds:
        preds = (val_probs >= t).astype(int)
        f1 = f1_score(y_val, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = t

    print(f"\nBest threshold (val): {best_thr}")
    print(f"Best validation F1: {best_f1}")

    # --- Final metrics using best threshold ---
    val_preds = (val_probs >= best_thr).astype(int)
    test_preds = (test_probs >= best_thr).astype(int)

    val_metrics = {
        "PR_AUC": float(val_pr_auc),
        "F1": float(f1_score(y_val, val_preds)),
        "Recall": float(recall_score(y_val, val_preds)),
        "Precision": float(precision_score(y_val, val_preds)),
    }

    test_metrics = {
        "PR_AUC": float(test_pr_auc),
        "F1": float(f1_score(y_test, test_preds)),
        "Recall": float(recall_score(y_test, test_preds)),
        "Precision": float(precision_score(y_test, test_preds)),
    }

    results = {
        "model_name": model_name,
        "best_iteration": model.get_best_iteration(),
        "best_threshold": float(best_thr),
        "validation": val_metrics,
        "test": test_metrics
    }

    # --- Save model ---
    joblib.dump(model, f"{save_dir}/{model_name}_model.pkl")

    # --- Save results ---
    with open(f"{save_dir}/{model_name}_metrics.json", "w") as f:
        json.dump(results, f, indent=4)

    print("\n--- Validation ---")
    print(val_metrics)
    print("\n--- Test ---")
    print(test_metrics)

    return model, results

In [20]:
# Small hyperparameter grid to try
param_grid = [None,
    {"depth": 6, "learning_rate": 0.05, "l2_leaf_reg": 3},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 5},
    {"depth": 8, "learning_rate": 0.05, "l2_leaf_reg": 5},
    {"depth": 8, "learning_rate": 0.03, "l2_leaf_reg": 7},
]

In [21]:
# SPECTER feature set
specter_features = specter_cols + stylometric_cols


In [22]:
# Grid search: keep the config with the best validation PR-AUC
best_result = None
best_score = 0
best_params = None

for i, params in enumerate(param_grid):
    print(f"\nRunning config {i+1}: {params}")
    
    _, results = train_and_evaluate(
        X_train[specter_features], y_train,
        X_val[specter_features], y_val,
        X_test[specter_features], y_test,
        model_name=f"specter_tuning_{i}",
        params=params
    )
    
    val_score = results["validation"]["PR_AUC"]
    
    if val_score > best_score:
        best_score = val_score
        best_result = results
        best_params = params

print("\nBest Params:", best_params)
print("Best Validation PR-AUC:", best_score)


Running config 1: None

===== Training specter_tuning_0 =====
0:	test: 0.5979802	best: 0.5979802 (0)	total: 958ms	remaining: 31m 54s
200:	test: 0.6981075	best: 0.6981075 (200)	total: 1m 11s	remaining: 10m 40s
400:	test: 0.7062837	best: 0.7063375 (399)	total: 2m 4s	remaining: 8m 17s
600:	test: 0.7101754	best: 0.7101754 (600)	total: 2m 55s	remaining: 6m 48s
800:	test: 0.7120034	best: 0.7121797 (793)	total: 3m 45s	remaining: 5m 37s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7121797098
bestIteration = 793

Shrink model to first 794 iterations.

Best threshold (val): 0.3801507537688442
Best validation F1: 0.432770588739395

--- Validation ---
{'PR_AUC': 0.37499189446644965, 'F1': 0.432770588739395, 'Recall': 0.6211562115621156, 'Precision': 0.3320620725933719}

--- Test ---
{'PR_AUC': 0.40362144683202617, 'F1': 0.4567901234567901, 'Recall': 0.5820660723649712, 'Precision': 0.37588892651540806}

Running config 2: {'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3

In [23]:
# Best params found
best_params 

In [ ]:
# Re-split with val folded into train (for the final fit)
import json

train_df, test_df, train_ids, test_ids = get_splits(df, val_split=False)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

assert len(train_df) == len(train_ids)
assert len(test_df) == len(test_ids)

print("Split integrity passed.")

Train: (124814, 786)
Test: (6958, 786)
Split integrity passed.


In [ ]:
# Build X/y (train+val vs test)
X_train, y_train, X_test, y_test = get_train_val_test_data(train_df=train_df, test_df=test_df, MODEL_COLS=MODEL_COLS, TARGET_COL=TARGET_COL)

print("X_train shape:", X_train.shape)

X_train shape: (124814, 774)


In [27]:
# Class balance
y_train.value_counts()

is_highly_cited
0    99424
1    25390
Name: count, dtype: int64

In [28]:
# Class weights to offset the imbalance
from collections import Counter

counter = Counter(y_train)
total = len(y_train)

class_weights = {
    0: total / (2 * counter[0]),
    1: total / (2 * counter[1])
}

print("Class weights:", class_weights)

Class weights: {0: 0.627685468297393, 1: 2.457936195352501}


In [29]:
# Helper: fit the final model on train+val at the frozen best config
def train_final_model(
    X_train_full, y_train_full,
    X_test, y_test,
    model_name,
    best_params,
    best_iteration,
    best_threshold,
    class_weights,
    save_dir="results"
):
    print(f"\n===== Final Training {model_name} (Train+Val) =====")

    final_params = {
        "iterations": best_iteration,   # freeze best iteration
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": 42,
        "verbose": 200,
        "class_weights": class_weights
    }

    # update with best hyperparams found during tuning
    final_params.update(best_params)

    model = CatBoostClassifier(**final_params)

    # Train on FULL data (train + val)
    model.fit(X_train_full, y_train_full)

    # ---- Test Evaluation ----
    test_probs = model.predict_proba(X_test)[:, 1]
    test_pr_auc = average_precision_score(y_test, test_probs)

    test_preds = (test_probs >= best_threshold).astype(int)

    test_metrics = {
        "PR_AUC": float(test_pr_auc),
        "F1": float(f1_score(y_test, test_preds)),
        "Recall": float(recall_score(y_test, test_preds)),
        "Precision": float(precision_score(y_test, test_preds)),
    }

    results = {
        "model_name": model_name,
        "best_iteration": best_iteration,
        "best_threshold": float(best_threshold),
        "test": test_metrics
    }

    # Save model
    joblib.dump(model, f"{save_dir}/{model_name}_final_model.pkl")

    # Save metrics
    with open(f"{save_dir}/{model_name}_final_metrics.json", "w") as f:
        json.dump(results, f, indent=4)

    print("\n--- Final Test Results ---")
    print(test_metrics)

    return model, results

In [31]:
# Best hyperparameters / iteration / threshold from the search
best_params = {
    "learning_rate": 0.05,
    "depth": 6,
    "l2_leaf_reg": 3
}
best_iteration = 793
best_threshold = 0.3801507537688442
iterations = 793

In [32]:
# Shape check
print("X_train shape:", X_train.shape)

X_train shape: (124814, 774)


In [33]:
# Fit the final deployed Value model (specter_final)
final_model, final_results = train_final_model(
    X_train[specter_features], y_train,
    X_test[specter_features], y_test,
    model_name="specter_final",
    best_params=best_params,
    best_iteration=best_iteration,
    best_threshold=best_threshold,
    class_weights=class_weights
)


===== Final Training specter_final (Train+Val) =====
0:	total: 251ms	remaining: 3m 19s
200:	total: 44.1s	remaining: 2m 9s
400:	total: 1m 26s	remaining: 1m 24s
600:	total: 2m 11s	remaining: 41.9s
792:	total: 2m 54s	remaining: 0us

--- Final Test Results ---
{'PR_AUC': 0.4232158950322031, 'F1': 0.46077405857740583, 'Recall': 0.9239643418982695, 'Precision': 0.3069151715728967}
